# Cross-Sectional Analysis — Hypothesis 1: Ex-ante Location Strategy · 2008–2019 Average

**Hypothesis 1 (Ex-ante)**: MNEs systematically avoid establishing affiliates in host countries with high historical disaster exposure; this penalty is attenuated when host-country governance quality is high.

**Strategy A**: `industry == "B-S_X_O"` (total aggregate, FATS outward)  
**Period**: 2008–2019 average cross-section (replaces the single 2008 snapshot to mitigate Global Financial Crisis distortion)  
**Main dependent variable**: `ln_avg_ent` = ln(avg(ENT, 2008–2019) + 1)  
**Estimator**: OLS with HC3 heteroskedasticity-robust standard errors

---

**Robustness checks:**
- **R1 — Average Employment**: Same specification with `ln_avg_emp = ln(avg(EMP, 2008–2019) + 1)`. Employment and affiliate counts should be positively correlated; this tests whether the disaster-exposure result is robust to the choice of MNE-presence variable.
- **R2 — Extensive Margin (Ever Active)**: Binary dependent variable `ever_active = 1` if the (home, host) pair had ENT > 0 in *at least one* year over 2008–2019, else 0. Estimated by LPM (OLS HC3) and Logit. This tests whether historical disaster exposure deters MNE *entry* altogether. **Limitation**: the 2008–2019 window means pairs active before 2008 appear as `ever_active = 1` regardless of whether disaster risk influenced their initial entry decision.

---

**Estimating equation (Model 3 — full):**

$$
\ln\!\left(\overline{\text{ENT}}_{ij}\right) = \alpha
+ \beta_1 \widetilde{ge}_{\text{host}}
+ \beta_2 \widetilde{\ln\!\left(\tfrac{\text{HistCnt}}{\text{pop}/10^6}\right)}_{\!\text{host}}
+ \beta_3 \!\left[\widetilde{ge}_{\text{host}} \times \widetilde{\ln\!\left(\tfrac{\text{HistCnt}}{\text{pop}/10^6}\right)}_{\!\text{host}}\right]
+ \mathbf{X}_{\text{macro}} \boldsymbol{\gamma}
+ \mathbf{Z}_{\text{gravity+geo}} \boldsymbol{\delta}
+ \varepsilon
$$

where $\overline{\text{ENT}}_{ij}$ is the 2008–2019 average affiliate count and $\tilde{x}$ denotes mean-centred $x$.

**Key design choices:**
- 2008–2019 **average** ENT instead of 2008 snapshot reduces Global Financial Crisis distortion.
- `hist_sev_static_host` (severity) **dropped** — correlated with per-capita count; count is the cleaner frequency proxy.
- Raw event count **normalised** by `pop_host / 1 000 000` (average population) to remove country-size bias.
- `ge_host` and the normalised count are **mean-centred** before forming the interaction term, eliminating structural VIF inflation.
- Two new geographic controls: `landlocked_host` (0/1 dummy) and `ln(surface_area_host)` capture host-country structural market-access and market-size characteristics.

## 0 · Imports

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.discrete.discrete_model import Logit
from pathlib import Path

_here = Path(os.getcwd()).resolve()
_root = _here.parent if _here.name == "notebooks" else _here
os.chdir(_root)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
print(f"Working directory: {os.getcwd()}")

Working directory: <repo>


In [2]:
# ── Table export utilities (for copy-paste into Word) ──────────────────────────
# Every results table below is exported as .csv (Excel) and standalone .html.
# To get a clean paste into Word: open the .html file in a browser, select the
# table, Ctrl+C, then Ctrl+V into Word — the browser puts real HTML on the
# clipboard, so Word creates a native formatted table instead of space-padded
# text. A combined "all_tables.html" with every table is built in the last cell,
# with an abbreviations legend at the top (copy that legend into your thesis'
# list of abbreviations / table notes).
import webbrowser

TABLES_DIR = Path("outputs/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
_ALL_TABLES = []  # (title, DataFrame) — collected for the combined export

# Short, table-ready variable labels (standard gravity/panel-regression notation)
VAR_LABELS = {
    'const':                          'Constant',
    'ln_gdp_pc_host':                 'ln GDPpc_host',
    'ln_pop_host':                    'ln Pop_host',
    'ln_gdp_pc_home':                 'ln GDPpc_home',
    'ln_pop_home':                    'ln Pop_home',
    'ln_distw':                       'ln Dist',
    'contig':                         'Contig',
    'colony':                         'Colony',
    'comlang_off':                    'Comlang',
    'landlocked_host':                'Landlocked_host',
    'ln_surface_area_host':           'ln Area_host',
    'ge_host_centered':               'GE_host',
    'ln_hist_cnt_pc_centered':        'ln HistFreq_host',
    'interaction_ge_hist_centered':   'GE_host x HistFreq_host',
    'hist_sev_total_centered':        'HistSev_host',
    'interaction_ge_hist_sev':        'GE_host x HistSev_host',
    
    'N':                              'N',
    'Adj. R²':                        'Adj. R²',
}

# Full definitions of the abbreviations above — printed as a legend in
# all_tables.html; copy this list into the thesis' abbreviations section.
ABBR_DEFINITIONS = [
    ('ln',                      'Natural logarithm (prefix)'),
    ('GDPpc_host / GDPpc_home', 'GDP per capita, constant 2015 USD, lagged one year (t-1)'),
    ('Pop_host / Pop_home',     'Population'),
    ('Dist',                    'Population-weighted bilateral distance (CEPII)'),
    ('Contig',                  'Shared land border (dummy)'),
    ('Colony',                  'Past colonial relationship (dummy)'),
    ('Comlang',                 'Common official language (dummy)'),
    ('Landlocked_host',         'Host country is landlocked (dummy)'),
    ('Area_host',                'Host country surface area, km²'),
    ('GE_host',                 'Government effectiveness, host country (WGI index, mean-centered)'),
    ('HistFreq_host',           'Historical disaster frequency per capita, host country, pre-2008 (log, mean-centered)'),
    ('HistSev_host',            'Historical disaster severity, host country, pre-2008 (log of affected-population share, mean-centered)'),
    ('GE_host x HistFreq_host', 'Interaction: governance effectiveness x historical disaster frequency'),
    ('GE_host x HistSev_host',  'Interaction: governance effectiveness x historical disaster severity'),
    ('N',                       'Number of observations'),
    ('Adj. R²',                 'Adjusted R-squared'),
]


def apply_labels(df):
    # Return a copy of df with its index relabeled via VAR_LABELS (thesis-ready names)
    out = df.copy()
    out.index = [VAR_LABELS.get(v, v) for v in out.index]
    out.index.name = df.index.name
    return out


def export_table(df, name, title=None):
    title = title or name
    csv_path  = TABLES_DIR / f"{name}.csv"
    html_path = TABLES_DIR / f"{name}.html"
    df.to_csv(csv_path)
    html_path.write_text(
        f"<html><head><meta charset='utf-8'><title>{title}</title></head>"
        f"<body><h2>{title}</h2>{df.to_html(border=1, na_rep='')}</body></html>",
        encoding="utf-8",
    )
    _ALL_TABLES.append((title, df))
    print(f"Exported '{name}': {csv_path.name}, {html_path.name}")


def stargazer_df(results, model_names, var_order=None, dep_var_label=''):
    # Same content as make_stargazer(), returned as a DataFrame with labeled variable names
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''
    all_vars = []
    for res in results:
        for v in res.params.index:
            if v not in all_vars:
                all_vars.append(v)
    if var_order is not None:
        all_vars = ([v for v in var_order if v in all_vars] +
                    [v for v in all_vars  if v not in var_order])
    rows = []
    for var in all_vars:
        label = VAR_LABELS.get(var, var)
        coef_row = {'Variable': label}
        pval_row = {'Variable': ''}
        for name, res in zip(model_names, results):
            if var in res.params:
                c, p = res.params[var], res.pvalues[var]
                coef_row[name] = f'{c:.4f}{stars(p)}'
                pval_row[name] = f'({p:.4f})'
            else:
                coef_row[name] = ''
                pval_row[name] = ''
        rows.append(coef_row)
        rows.append(pval_row)
    rows.append({'Variable': 'N',       **{n: f'{int(r.nobs):,}'      for n, r in zip(model_names, results)}})
    rows.append({'Variable': 'Adj. R²', **{n: f'{r.rsquared_adj:.4f}' for n, r in zip(model_names, results)}})
    out = pd.DataFrame(rows).set_index('Variable')
    out.columns.name = f'Dep. var: {dep_var_label}'
    return out


def ame_table_df(models_me, model_names, dep_var_label='ever_active (Logit AME)'):
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''
    all_regs = []
    for _, regs in models_me:
        for r in regs:
            if r not in all_regs:
                all_regs.append(r)
    rows = []
    for var in all_regs:
        label = VAR_LABELS.get(var, var)
        coef_row = {'Variable': label}
        pval_row = {'Variable': ''}
        for name, (me, regs) in zip(model_names, models_me):
            if var in regs:
                idx = regs.index(var)
                coef_row[name] = f'{me.margeff[idx]:.4f}{stars(me.pvalues[idx])}'
                pval_row[name] = f'({me.pvalues[idx]:.4f})'
            else:
                coef_row[name] = ''
                pval_row[name] = ''
        rows.append(coef_row)
        rows.append(pval_row)
    out = pd.DataFrame(rows).set_index('Variable')
    out.columns.name = f'Dep. var: {dep_var_label}'
    return out


def build_master_html():
    # Combine every exported table into one HTML page for easy browsing/copying,
    # preceded by a legend of the abbreviations used in the tables.
    legend_rows = "".join(
        f"<tr><td>{abbr}</td><td>{definition}</td></tr>"
        for abbr, definition in ABBR_DEFINITIONS
    )
    parts = ["<html><head><meta charset='utf-8'><title>Regression tables</title></head><body>",
             "<p>Open this file in a browser, select each table, Ctrl+C, then "
             "Ctrl+V into Word to get a native formatted table.</p>",
             "<h2>Abbreviations</h2>",
             f"<table border='1'><tr><th>Abbreviation</th><th>Definition</th></tr>{legend_rows}</table>"]
    for title, tdf in _ALL_TABLES:
        parts.append(f"<h2>{title}</h2>")
        parts.append(tdf.to_html(border=1, na_rep=""))
    parts.append("</body></html>")
    out_path = TABLES_DIR / "all_tables.html"
    out_path.write_text("\n".join(parts), encoding="utf-8")
    print(f"Combined file: {out_path.resolve()}")
    return out_path


print(f"Tables will be exported to: {TABLES_DIR.resolve()}")


Tables will be exported to: <repo>\outputs\tables


## 1 · Data Preparation (2008–2019 Average)

The raw estimation sample (`estimation_sample.parquet`) is a 12-year panel. We collapse it to a pair-level cross-section by averaging time-varying controls across 2008–2019. Time-invariant variables (historical exposure, gravity, geography) keep their first observed value (identical across years for a given pair).

In [3]:
# ── Load estimation sample (intensive margin) ────────────────────────────────
raw = pd.read_parquet("data/Silver/estimation_sample.parquet")

# Filter: B-S_X_O aggregate, 2008-2019
bsxo = raw.loc[
    (raw["industry"] == "B-S_X_O") & raw["year"].between(2008, 2019)
].copy()
print(f"B-S_X_O panel: {len(bsxo):,} rows  "
      f"({bsxo['home_country'].nunique()} home × "
      f"{bsxo['host_country'].nunique()} host × "
      f"{bsxo['year'].nunique()} years)")

# ── Collapse to pair-level cross-section ─────────────────────────────────────
# time-varying → mean (NaN-safe); time-invariant → first (same for all years)
TIME_VARYING = [
    "ent", "emp", "tur",
    "ln_gdp_pc_host", "pop_host", "ge_host",
    "ln_gdp_pc_home", "pop_home", "ge_home",
]
TIME_INVARIANT = [
    "home_region",
    "hist_sev_static_host",  "hist_cnt_total_static_host",
    "hist_sev_static_home",  "hist_cnt_total_static_home",
    "distw", "contig", "colony", "comlang_off",
    "landlocked_host", "surface_area_host",
]

agg_dict = {col: "mean"  for col in TIME_VARYING   if col in bsxo.columns}
agg_dict.update({col: "first" for col in TIME_INVARIANT if col in bsxo.columns})

df = (
    bsxo
    .groupby(["home_country", "host_country"], as_index=False)
    .agg(agg_dict)
)

print(f"\nCollapsed cross-section: {len(df):,} pairs "
      f"({df['home_country'].nunique()} home × {df['host_country'].nunique()} host)")
print(f"\nAverage ENT (2008–2019) — descriptive stats:")
print(df["ent"].describe().round(3))
print(f"\n  Pairs with avg_ent = NaN (all years suppressed): {df['ent'].isna().sum()}")
print(f"  Pairs with avg_ent = 0   (never any affiliates):  {(df['ent'] == 0).sum()}")

B-S_X_O panel: 26,000 rows  (29 home × 185 host × 12 years)

Collapsed cross-section: 2,472 pairs (29 home × 185 host)

Average ENT (2008–2019) — descriptive stats:
count   2472.0000
mean      73.7840
std      266.5500
min        0.0830
25%        1.5960
50%        6.8590
75%       31.0280
max     4115.3330
Name: ent, dtype: float64

  Pairs with avg_ent = NaN (all years suppressed): 0
  Pairs with avg_ent = 0   (never any affiliates):  0


## 2 · Variable Transformations

Key steps:
1. **Dependent variables**: `ln_avg_ent = ln(avg_ent + 1)` and `ln_avg_emp = ln(avg_emp + 1)`.
2. **Log-transforms** for skewed continuous controls (GDP per capita, population, distance, surface area).
3. **Disaster density**: `hist_cnt_total_static_host` (pre-2008 event count) divided by `pop_host / 1 000 000` before logging — removes the country-size bias from raw EM-DAT accumulation.
4. **Mean-centring** of `ge_host` and `ln_hist_cnt_pc` before forming the interaction term (eliminates structural VIF inflation without changing slope coefficients).
5. **Geographic controls**: `landlocked_host` (0/1, no transformation needed) and `ln(surface_area_host)` (log-transform of km²).

In [4]:
# ── Dependent variables ───────────────────────────────────────────────────────
df["ln_avg_ent"] = np.log1p(df["ent"])
df["ln_avg_emp"] = np.log1p(df["emp"])

# ── Log-transforms for continuous controls ────────────────────────────────────
# ln_gdp_pc_host / ln_gdp_pc_home already pre-computed (lagged t-1) in script 08
for col in ["pop_host", "pop_home", "distw"]:
    df[f"ln_{col}"] = np.log(df[col])
    n_nan = df[f"ln_{col}"].isna().sum()
    if n_nan > 0:
        print(f"  WARNING ln_{col}: {n_nan} NaN after log")

# Surface area (km²) — log-transform, right-skewed
df["ln_surface_area_host"] = np.log(df["surface_area_host"])

# ── Historical disaster DENSITY (per-million-people) ──────────────────────────
# Dividing by pop_host/1e6 removes the country-size bias: large countries
# accumulate more EM-DAT entries AND attract more FDI, confounding hazard with size.
# Average pop_host (2008-2019) is used as the scale denominator.
df["disasters_per_million"] = (
    df["hist_cnt_total_static_host"] / (df["pop_host"] / 1_000_000)
)
df["ln_hist_cnt_pc"] = np.log1p(df["disasters_per_million"])

# ── Mean-centring to eliminate interaction VIF inflation ──────────────────────
df["ge_host_centered"]        = df["ge_host"]        - df["ge_host"].mean()
df["ln_hist_cnt_pc_centered"] = df["ln_hist_cnt_pc"] - df["ln_hist_cnt_pc"].mean()
df["ln_gdp_pc_host_centered"] = df["ln_gdp_pc_host"] - df["ln_gdp_pc_host"].mean()

# ── Interaction (H1: beta_3 > 0 => governance attenuates the disaster penalty) ─
df["interaction_ge_hist_centered"] = (
    df["ge_host_centered"] * df["ln_hist_cnt_pc_centered"]
)


# ── Diagnostics ───────────────────────────────────────────────────────────────
print("Transformed variables — descriptive stats:")
check_cols = [
    "ln_avg_ent", "ln_avg_emp",
    "ln_gdp_pc_host", "ln_pop_host", "ln_gdp_pc_home", "ln_pop_home", "ln_distw",
    "landlocked_host", "ln_surface_area_host",
    "ge_host_centered", "ln_hist_cnt_pc_centered", "interaction_ge_hist_centered",
]
print(df[check_cols].describe().T[["count", "mean", "std", "min", "max"]].round(3))


Transformed variables — descriptive stats:
                                 count    mean    std     min     max
ln_avg_ent                   2472.0000  2.3970 1.7630  0.0800  8.3230
ln_avg_emp                   2337.0000  5.3830 3.6200  0.0000 13.7510
ln_gdp_pc_host               2472.0000  9.0620 1.3690  5.5510 11.6340
ln_pop_host                  2472.0000 16.1940 1.9160  9.2760 21.0470
ln_gdp_pc_home               2472.0000 10.3920 0.5960  8.7900 11.5800
ln_pop_home                  2472.0000 16.2810 1.3420 12.9600 18.2310
ln_distw                     2472.0000  8.1770 0.9690  5.0810  9.8790
landlocked_host              2472.0000  0.1790 0.3830  0.0000  1.0000
ln_surface_area_host         2472.0000 11.8220 2.4350  2.9960 16.6110
ge_host_centered             2472.0000 -0.0000 0.9520 -2.3510  1.9230
ln_hist_cnt_pc_centered      2472.0000  0.0000 0.8510 -1.1850  3.6990
interaction_ge_hist_centered 2472.0000  0.0390 0.7240 -3.5800  3.0560


## 3 · Multicollinearity Analysis (VIF)

VIF is computed on the **Model 3 regressor set** using the complete-case sample. Values above 5 are flagged.

> **Note on `ln_pop_host` vs `ln_surface_area_host`**: both capture country size from different angles (demographic vs. physical). They may be moderately correlated; the VIF check flags if co-linearity is problematic.

In [5]:
M3_regressors = [
    "ln_gdp_pc_host",       "ln_pop_host",
    "ln_gdp_pc_home",       "ln_pop_home",
    "ln_distw",
    "contig",               "colony",             "comlang_off",
    "landlocked_host",      "ln_surface_area_host",
    "ge_host_centered",
] +  [
    "ln_hist_cnt_pc_centered",
    "interaction_ge_hist_centered",
]

vif_cols = ["ln_avg_ent"] + M3_regressors
vif_df   = df[vif_cols].dropna()
print(f"VIF computed on {len(vif_df):,} complete observations")

X_vif = sm.add_constant(vif_df[M3_regressors].values, prepend=True)
vif_results = pd.DataFrame({
    "Variable": M3_regressors,
    "VIF": [variance_inflation_factor(X_vif, i + 1) for i in range(len(M3_regressors))],
}).set_index("Variable")

vif_results["VIF"]  = vif_results["VIF"].round(4)
vif_results["Flag"] = np.where(vif_results["VIF"] > 5, ">> HIGH VIF", "")

print("\nVariance Inflation Factors — Model 3 regressors")
print("=" * 55)
print(vif_results.to_string())
print("=" * 55)

n_high = (vif_results["VIF"] > 5).sum()
if n_high == 0:
    print("\n✓ No variables flagged (all VIF ≤ 5).")
else:
    flagged = vif_results[vif_results["VIF"] > 5].index.tolist()
    print(f"\n⚠ {n_high} variable(s) flagged (VIF > 5): {flagged}")

export_table(apply_labels(vif_results), "vif_main", "VIF — Model 3 regressors (main spec)")


VIF computed on 2,472 complete observations

Variance Inflation Factors — Model 3 regressors
                                VIF Flag
Variable                                
ln_gdp_pc_host               4.6882     
ln_pop_host                  4.2735     
ln_gdp_pc_home               1.0920     
ln_pop_home                  1.0863     
ln_distw                     1.5993     
contig                       1.2687     
colony                       1.5998     
comlang_off                  1.6042     
landlocked_host              1.1375     
ln_surface_area_host         3.1275     
ge_host_centered             4.5279     
ln_hist_cnt_pc_centered      2.0862     
interaction_ge_hist_centered 1.1443     

✓ No variables flagged (all VIF ≤ 5).
Exported 'vif_main': vif_main.csv, vif_main.html


## 4 · OLS Regressions — Main Specification (ln\_avg\_ent)

Three nested models, each using listwise deletion on its own regressor set + `ln_avg_ent`:

| Model | Regressors added |
|---|---|
| **M1** Gravity + Governance + Non-linear terms | `ln_gdp_pc_host`, `ln_pop_host`, `ln_gdp_pc_home`, `ln_pop_home`, `ln_distw`, `contig`, `colony`, `comlang_off`, `landlocked_host`, `ln_surface_area_host`, `ge_host_centered`, `ge_host_sq`, `ln_gdp_pc_host_sq`, `interaction_ge_gdppc` |
| **M2** + Disaster Density | M1 + `ln_hist_cnt_pc_centered` |
| **M3** + Institutional Interaction | M2 + `interaction_ge_hist_centered` |

**Non-linear terms** (`ge_host_sq`, `ln_gdp_pc_host_sq`, `interaction_ge_gdppc`) test whether the governance effect is non-monotonic and whether it depends on the host country's development level; both source variables are mean-centred before squaring/interacting to limit VIF inflation. These terms are specific to the **main specification** and are not carried into the robustness checks (R1, R2, Section 9) below.

In [6]:
def run_ols_hc3(data, dep_var, regressors):
    # OLS with HC3 robust standard errors
    cols = [dep_var] + regressors
    sub  = data[cols].dropna()
    y    = sub[dep_var]
    X    = sm.add_constant(sub[regressors], prepend=True)
    return sm.OLS(y, X).fit(cov_type="HC3")


M1_regressors = [
    "ln_gdp_pc_host",  "ln_pop_host",
    "ln_gdp_pc_home",  "ln_pop_home",
    "ln_distw",
    "contig",          "colony",             "comlang_off",
    "landlocked_host", "ln_surface_area_host",
    "ge_host_centered",
]
M2_regressors     = M1_regressors + ['ln_hist_cnt_pc_centered']
M3_regressors_full = M2_regressors + ['interaction_ge_hist_centered']

# ── Main-specification-only regressor sets (adds GE², ln_GDPpc², GE x ln_GDPpc) ──
# Kept separate from M1_regressors/M2_regressors/M3_regressors_full above so the
# robustness checks (R1, R2, Section 9) keep using the base specification.
M1_regressors_main = M1_regressors 
M2_regressors_main = M1_regressors_main + ['ln_hist_cnt_pc_centered']
M3_regressors_main = M2_regressors_main + ['interaction_ge_hist_centered']

res_m1 = run_ols_hc3(df, "ln_avg_ent", M1_regressors_main)
res_m2 = run_ols_hc3(df, "ln_avg_ent", M2_regressors_main)
res_m3 = run_ols_hc3(df, "ln_avg_ent", M3_regressors_main)

print("Main models — dep. var: ln_avg_ent (2008–2019 average):")
print(f"  M1: N={int(res_m1.nobs):,}  Adj.R²={res_m1.rsquared_adj:.4f}")
print(f"  M2: N={int(res_m2.nobs):,}  Adj.R²={res_m2.rsquared_adj:.4f}")
print(f"  M3: N={int(res_m3.nobs):,}  Adj.R²={res_m3.rsquared_adj:.4f}")


Main models — dep. var: ln_avg_ent (2008–2019 average):
  M1: N=2,472  Adj.R²=0.6098
  M2: N=2,472  Adj.R²=0.6166
  M3: N=2,472  Adj.R²=0.6192


## 5 · Consolidated Regression Table — Main Specification

In [7]:
def make_stargazer(results, model_names, var_order=None, dep_var_label=''):
    # Stargazer-style table: coef (stars) + (p-value)
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''
    all_vars = []
    for res in results:
        for v in res.params.index:
            if v not in all_vars:
                all_vars.append(v)
    if var_order is not None:
        all_vars = ([v for v in var_order if v in all_vars] +
                    [v for v in all_vars  if v not in var_order])
    col_w, lbl_w = 20, 38
    sep = '─' * (lbl_w + col_w * len(results))
    lines = [f'Dep. var: {dep_var_label}', sep,
             f"{'':>{lbl_w}}" + ''.join(f'{n:>{col_w}}' for n in model_names), sep]
    for var in all_vars:
        coef_row = f'{var:<{lbl_w}}'
        pval_row = f"{'':>{lbl_w}}"
        for res in results:
            if var in res.params:
                c, p = res.params[var], res.pvalues[var]
                coef_row += f"{f'{c:.4f}{stars(p)}':>{col_w}}"
                pval_row += f"{f'({p:.4f})':>{col_w}}"
            else:
                coef_row += f"{'':>{col_w}}"
                pval_row += f"{'':>{col_w}}"
        lines += [coef_row, pval_row]
    lines += [sep,
              f"{'N':<{lbl_w}}" + ''.join(f'{int(r.nobs):>{col_w},}' for r in results),
              'Adj. R²'.ljust(lbl_w) + ''.join(f'{r.rsquared_adj:.4f}'.rjust(col_w) for r in results),
              sep,
              'Significance: * p<0.10  ** p<0.05  *** p<0.01  |  p-values in parentheses (HC3 robust)']
    return '\n'.join(lines)


DISPLAY_ORDER = [
    'const',
    'ln_gdp_pc_host',       'ln_pop_host',
    'ln_gdp_pc_home',       'ln_pop_home',
    'ln_distw',             'contig',        'colony',       'comlang_off',
    'landlocked_host',      'ln_surface_area_host',
    'ge_host_centered',
   
    'ln_hist_cnt_pc_centered',
    'interaction_ge_hist_centered',
]

print(make_stargazer(
    [res_m1, res_m2, res_m3],
    ['Model 1', 'Model 2', 'Model 3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ln_avg_ent  (2008–19 average)'
))

df_stargazer_main = stargazer_df(
    [res_m1, res_m2, res_m3],
    ['Model 1', 'Model 2', 'Model 3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ln_avg_ent (2008–19 average)'
)
export_table(df_stargazer_main, "regression_main_ln_avg_ent",
             "Main Specification — ln_avg_ent (2008–2019 average)")


Dep. var: ln_avg_ent  (2008–19 average)
──────────────────────────────────────────────────────────────────────────────────────────────────
                                                   Model 1             Model 2             Model 3
──────────────────────────────────────────────────────────────────────────────────────────────────
const                                          -22.7654***         -24.0036***         -24.6091***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_host                                   0.3051***           0.3327***           0.3696***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_pop_host                                      0.3617***           0.4422***           0.4525***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_home                                   1.0506***           

## 6 · Robustness I — Average Employment (ln\_avg\_emp)

**Rationale**: ENT counts legal entities (affiliates); EMP measures operational scale (workers). Both reflect the same underlying location decision — more affiliates should employ more workers. If H1 captures a genuine location-strategy channel, the sign pattern should replicate. A divergence would suggest the result is driven by affiliate-count mechanics (e.g., single-employee holding structures) rather than substantive economic presence.

**Expected outcome**: signs and significance on `ln_hist_cnt_pc_centered`, `ge_host_centered`, and `interaction_ge_hist_centered` should closely mirror the main specification.

In [8]:
res_r1_m1 = run_ols_hc3(df, "ln_avg_emp", M1_regressors)
res_r1_m2 = run_ols_hc3(df, "ln_avg_emp", M2_regressors)
res_r1_m3 = run_ols_hc3(df, "ln_avg_emp", M3_regressors_full)

print('Robustness I — dep. var: ln_avg_emp (2008–19 average):')
print(f'  M1: N={int(res_r1_m1.nobs):,}  Adj.R²={res_r1_m1.rsquared_adj:.4f}')
print(f'  M2: N={int(res_r1_m2.nobs):,}  Adj.R²={res_r1_m2.rsquared_adj:.4f}')
print(f'  M3: N={int(res_r1_m3.nobs):,}  Adj.R²={res_r1_m3.rsquared_adj:.4f}')

print()
print(make_stargazer(
    [res_r1_m1, res_r1_m2, res_r1_m3],
    ['R1-M1', 'R1-M2', 'R1-M3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ln_avg_emp  (2008–19 average)'
))

df_stargazer_r1 = stargazer_df(
    [res_r1_m1, res_r1_m2, res_r1_m3],
    ['R1-M1', 'R1-M2', 'R1-M3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ln_avg_emp (2008–19 average)'
)
export_table(df_stargazer_r1, "regression_r1_ln_avg_emp",
             "Robustness I — ln_avg_emp (2008–2019 average)")

Robustness I — dep. var: ln_avg_emp (2008–19 average):
  M1: N=2,337  Adj.R²=0.5804
  M2: N=2,337  Adj.R²=0.5819
  M3: N=2,337  Adj.R²=0.5836

Dep. var: ln_avg_emp  (2008–19 average)
──────────────────────────────────────────────────────────────────────────────────────────────────
                                                     R1-M1               R1-M2               R1-M3
──────────────────────────────────────────────────────────────────────────────────────────────────
const                                          -49.1197***         -50.4044***         -51.4036***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_host                                   0.5773***           0.6052***           0.6665***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_pop_host                                      0.8161***           0.8985***           0.9160***
                         

## 7 · Robustness II — Extensive Margin: `ever_active` Dummy

**Rationale**: ENT and EMP measure the *intensive* margin — scale of presence *among* investing pairs. A complementary question is whether historical disaster exposure deters MNE *entry* altogether — the *extensive* margin.

**Construction**: From `estimation_sample_extensive.parquet` — a Cartesian panel of all (home, host) pairs with macro-data coverage (29 EU/EFTA home × ~190 host × 12 years, B-S_X_O only) — we collapse to pair level and define:

$$
\text{ever\_active}_{ij} = \mathbf{1}\!\left[\,\text{ENT}_{ijt} > 0 \text{ for at least one } t \in \{2008,\ldots,2019\}\,\right]
$$

**Estimators**:
- **LPM** (OLS with HC3): linear probability model, directly comparable to the main specification.
- **Logit**: accounts for the binary outcome; average marginal effects (AME, evaluated at the mean) are reported.

**Limitation**: `ever_active` captures presence at *any point* in 2008–2019, not new-entry decisions during the period. Pairs established before 2008 that remained active appear as `ever_active = 1` regardless of whether disaster risk influenced their initial entry. This window-censoring caveat constrains causal inference about the entry-deterrence channel.

In [9]:
# ── Load and collapse extensive sample to pair level ─────────────────────────
ext_raw = pd.read_parquet("data/Silver/estimation_sample_extensive.parquet")
print(f'Extensive sample loaded: {ext_raw.shape}')

EXT_TIME_VARYING   = ['ln_gdp_pc_host', 'pop_host', 'ge_host',
                      'ln_gdp_pc_home', 'pop_home', 'ge_home']
EXT_TIME_INVARIANT = [
    'home_region', '_fats_active',
    'hist_sev_static_host',  'hist_cnt_total_static_host',
    'hist_sev_static_home',  'hist_cnt_total_static_home',
    'distw', 'contig', 'colony', 'comlang_off',
    'landlocked_host', 'surface_area_host',
]

ext_agg = {col: 'mean'  for col in EXT_TIME_VARYING   if col in ext_raw.columns}
ext_agg.update({col: 'first' for col in EXT_TIME_INVARIANT if col in ext_raw.columns})

df_ext = (
    ext_raw
    .groupby(['home_country', 'host_country'], as_index=False)
    .agg(ext_agg)
)

# Binary dep var: 1 if pair ever had ENT>0 in 2008-2019 (from _fats_active flag)
df_ext['ever_active'] = df_ext['_fats_active'].astype(int)

print(f'\nPair-level extensive cross-section: {len(df_ext):,} pairs '
      f'({df_ext["home_country"].nunique()} home x {df_ext["host_country"].nunique()} host)')
print(f'  ever_active = 1 (at least 1 year with ENT>0):  {df_ext["ever_active"].sum():,}')
print(f'  ever_active = 0 (never active 2008-2019):       {(df_ext["ever_active"] == 0).sum():,}')
print(f'  Share ever-active: {df_ext["ever_active"].mean():.3f}')

Extensive sample loaded: (65772, 60)

Pair-level extensive cross-section: 5,481 pairs (29 home x 190 host)
  ever_active = 1 (at least 1 year with ENT>0):  2,476
  ever_active = 0 (never active 2008-2019):       3,005
  Share ever-active: 0.452


In [10]:
# ── Variable transformations for extensive sample ────────────────────────────
# ln_gdp_pc_host / ln_gdp_pc_home already pre-computed (lagged t-1) in script 09
for col in ['pop_host', 'pop_home', 'distw']:
    df_ext[f'ln_{col}'] = np.log(df_ext[col])

df_ext['ln_surface_area_host'] = np.log(df_ext['surface_area_host'])

# Disaster density — same construction as intensive sample
df_ext['disasters_per_million'] = (
    df_ext['hist_cnt_total_static_host'] / (df_ext['pop_host'] / 1_000_000)
)
df_ext['ln_hist_cnt_pc'] = np.log1p(df_ext['disasters_per_million'])

# Mean-centring w.r.t. extensive-sample means (own-sample centering)
df_ext['ge_host_centered']        = df_ext['ge_host']        - df_ext['ge_host'].mean()
df_ext['ln_hist_cnt_pc_centered'] = df_ext['ln_hist_cnt_pc'] - df_ext['ln_hist_cnt_pc'].mean()
df_ext['interaction_ge_hist_centered'] = (
    df_ext['ge_host_centered'] * df_ext['ln_hist_cnt_pc_centered']
)

print('Extensive sample — transformed variables:')
ext_check = [
    'ever_active',
    'ln_gdp_pc_host', 'ln_pop_host', 'ln_gdp_pc_home', 'ln_pop_home', 'ln_distw',
    'landlocked_host', 'ln_surface_area_host',
    'ge_host_centered', 'ln_hist_cnt_pc_centered', 'interaction_ge_hist_centered',
]
print(df_ext[[c for c in ext_check if c in df_ext.columns]]
      .describe().T[['count', 'mean', 'std', 'min', 'max']].round(3))

Extensive sample — transformed variables:
                                 count    mean    std     min     max
ever_active                  5481.0000  0.4520 0.4980  0.0000  1.0000
ln_gdp_pc_host               5481.0000  8.6290 1.4240  5.5510 11.6340
ln_pop_host                  5481.0000 15.5310 2.2280  9.2760 21.0360
ln_gdp_pc_home               5481.0000 10.1450 0.6690  8.8400 11.5790
ln_pop_home                  5481.0000 15.8700 1.3660 12.9950 18.2190
ln_distw                     5481.0000  8.4610 0.8490  5.0810  9.8800
landlocked_host              5481.0000  0.2000 0.4000  0.0000  1.0000
ln_surface_area_host         5481.0000 11.2220 2.7590  2.9960 16.6110
ge_host_centered             5481.0000 -0.0000 0.9280 -2.0530  2.2010
ln_hist_cnt_pc_centered      5481.0000  0.0000 1.0920 -1.4460  3.7840
interaction_ge_hist_centered 5481.0000  0.0270 0.8860 -2.4840  3.6150


In [11]:
# ── LPM (OLS HC3) ─────────────────────────────────────────────────────────────
res_r2_m1 = run_ols_hc3(df_ext, 'ever_active', M1_regressors)
res_r2_m2 = run_ols_hc3(df_ext, 'ever_active', M2_regressors)
res_r2_m3 = run_ols_hc3(df_ext, 'ever_active', M3_regressors_full)

print('Robustness II — LPM (dep. var: ever_active):')
print(f'  M1: N={int(res_r2_m1.nobs):,}  Adj.R²={res_r2_m1.rsquared_adj:.4f}')
print(f'  M2: N={int(res_r2_m2.nobs):,}  Adj.R²={res_r2_m2.rsquared_adj:.4f}')
print(f'  M3: N={int(res_r2_m3.nobs):,}  Adj.R²={res_r2_m3.rsquared_adj:.4f}')

print()
print(make_stargazer(
    [res_r2_m1, res_r2_m2, res_r2_m3],
    ['LPM-M1', 'LPM-M2', 'LPM-M3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ever_active  (LPM)'
))

df_stargazer_r2_lpm = stargazer_df(
    [res_r2_m1, res_r2_m2, res_r2_m3],
    ['LPM-M1', 'LPM-M2', 'LPM-M3'],
    var_order=DISPLAY_ORDER,
    dep_var_label='ever_active (LPM)'
)
export_table(df_stargazer_r2_lpm, "regression_r2_lpm_ever_active",
             "Robustness II — ever_active, LPM")

Robustness II — LPM (dep. var: ever_active):


  M1: N=5,481  Adj.R²=0.4063
  M2: N=5,481  Adj.R²=0.4084
  M3: N=5,481  Adj.R²=0.4087

Dep. var: ever_active  (LPM)
──────────────────────────────────────────────────────────────────────────────────────────────────
                                                    LPM-M1              LPM-M2              LPM-M3
──────────────────────────────────────────────────────────────────────────────────────────────────
const                                           -4.0905***          -4.2316***          -4.2665***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_host                                   0.0616***           0.0687***           0.0714***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_pop_host                                      0.0651***           0.0748***           0.0754***
                                                  (0.0000)            (0.0000)            (

In [12]:
# ── Logit (Average Marginal Effects) ─────────────────────────────────────────
def run_logit_ame(data, dep_var, regressors):
    # Fit Logit; return (result, margeff_result)
    cols = [dep_var] + regressors
    sub  = data[cols].dropna()
    y    = sub[dep_var].values.astype(float)
    X    = sm.add_constant(sub[regressors].values.astype(float), prepend=True)
    res  = Logit(y, X).fit(disp=False)
    me   = res.get_margeff()
    return res, me


res_logit_m1, me_m1 = run_logit_ame(df_ext, 'ever_active', M1_regressors)
res_logit_m2, me_m2 = run_logit_ame(df_ext, 'ever_active', M2_regressors)
res_logit_m3, me_m3 = run_logit_ame(df_ext, 'ever_active', M3_regressors_full)

print('Logit convergence & fit:')
print(f'  M1: N={int(res_logit_m1.nobs):,}  pseudo-R²={res_logit_m1.prsquared:.4f}')
print(f'  M2: N={int(res_logit_m2.nobs):,}  pseudo-R²={res_logit_m2.prsquared:.4f}')
print(f'  M3: N={int(res_logit_m3.nobs):,}  pseudo-R²={res_logit_m3.prsquared:.4f}')


def ame_table(models_me, model_names):
    # Side-by-side AME table for multiple logit models
    # models_me: list of (me_result, regressors_list) tuples
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''
    # union of all regressors, in M3 order
    all_regs = []
    for _, regs in models_me:
        for r in regs:
            if r not in all_regs:
                all_regs.append(r)
    col_w, lbl_w = 22, 38
    sep = '─' * (lbl_w + col_w * len(models_me))
    lines = ['Dep. var: ever_active  (Logit AME)', sep,
             f"{'Variable':<{lbl_w}}" + ''.join(f'{n:>{col_w}}' for n in model_names), sep]
    for var in all_regs:
        coef_row = f'{var:<{lbl_w}}'
        pval_row = f"{'':>{lbl_w}}"
        for me, regs in models_me:
            if var in regs:
                idx = regs.index(var)
                me_val = me.margeff[idx]
                p_val  = me.pvalues[idx]
                coef_row += f"{f'{me_val:.4f}{stars(p_val)}':>{col_w}}"
                pval_row += f"{f'({p_val:.4f})':>{col_w}}"
            else:
                coef_row += f"{'':>{col_w}}"
                pval_row += f"{'':>{col_w}}"
        lines += [coef_row, pval_row]
    lines += [sep,
              'Significance: * p<0.10  ** p<0.05  *** p<0.01  |  p-values in parentheses',
              'AME = average marginal effects evaluated at the mean']
    return '\n'.join(lines)


print()
print(ame_table(
    [(me_m1, M1_regressors), (me_m2, M2_regressors), (me_m3, M3_regressors_full)],
    ['Logit-M1 AME', 'Logit-M2 AME', 'Logit-M3 AME'],
))

df_ame = ame_table_df(
    [(me_m1, M1_regressors), (me_m2, M2_regressors), (me_m3, M3_regressors_full)],
    ['Logit-M1 AME', 'Logit-M2 AME', 'Logit-M3 AME'],
)
export_table(df_ame, "regression_r2_logit_ame", "Robustness II — ever_active, Logit AME")

Logit convergence & fit:
  M1: N=5,481  pseudo-R²=0.3803
  M2: N=5,481  pseudo-R²=0.3842
  M3: N=5,481  pseudo-R²=0.3845

Dep. var: ever_active  (Logit AME)
────────────────────────────────────────────────────────────────────────────────────────────────────────
Variable                                        Logit-M1 AME          Logit-M2 AME          Logit-M3 AME
────────────────────────────────────────────────────────────────────────────────────────────────────────
ln_gdp_pc_host                                     0.0535***             0.0623***             0.0648***
                                                    (0.0000)              (0.0000)              (0.0000)
ln_pop_host                                        0.0633***             0.0767***             0.0777***
                                                    (0.0000)              (0.0000)              (0.0000)
ln_gdp_pc_home                                     0.2332***             0.2342***             0.2345***
   

## 8 · Findings Summary

### Design Changes vs. Previous Version

| Change | Rationale |
|---|---|
| 2008 snapshot → 2008–2019 **average** | 2008 ENT reflects the pre-GFC peak; averaging gives a more stable measure of equilibrium location strategies |
| Added `landlocked_host` | Market-access barrier: landlocked countries face higher trade costs, structurally fewer affiliates |
| Added `ln_surface_area_host` | Physical market size, orthogonal to population density |

---

### Main Specification: ln\_avg\_ent

*[To be completed after running the notebook.]*

Key variables to interpret:

| Variable | H1 prediction | M2 | M3 | Interpretation |
|---|---|---|---|---|
| `ln_hist_cnt_pc_centered` | **−** (disaster penalty) | — | — | Does historical exposure reduce average affiliate counts? |
| `ge_host_centered` | **+** (institutional pull) | — | — | Better governance attracts more affiliates |
| `interaction_ge_hist_centered` | **+** (attenuation) | — | — | Does governance attenuate the disaster penalty? |
| `landlocked_host` | **−** (market-access barrier) | — | — | Landlocked hosts attract fewer affiliates |
| `ln_surface_area_host` | **+** (market size) | — | — | Larger host markets attract more affiliates |

---

### Robustness I: ln\_avg\_emp

If disaster-exposure signs replicate across ENT and EMP, the result reflects substantive MNE presence rather than affiliate-count mechanics. The two measures should be highly correlated at the pair level (Pearson r typically > 0.9 in FATS data).

---

### Robustness II: ever\_active (Extensive Margin)

The extensive-margin test asks whether disaster exposure deters entry *at all*. H1 predicts a **negative** coefficient on `ln_hist_cnt_pc_centered`: high historical exposure lowers the probability of ever observing any affiliates.

**Window-censoring caveat**: `ever_active = 1` captures presence at *any point* in 2008–2019, not *new* entry. Pre-2008 pairs that stayed active throughout inflate the `ever_active = 1` group and likely bias the coefficient toward zero (or positive). The LPM and Logit results should therefore be interpreted as a lower bound on the true entry-deterrence effect.

---

> **Note on historical severity collinearity (carried forward)**: `hist_sev_static_host` and `hist_sev_dynamic_host` remain perfectly collinear in any single-year cross-section, and are highly correlated with `hist_cnt_total_static_host` even at the averaged level. Both severity measures are dropped from all specifications; the per-capita frequency count `ln_hist_cnt_pc_centered` is the sole disaster-exposure variable.

## 9 · Alternative Exposure Proxy — Severity-based Historical Exposure (ln\_avg\_ent)

The main specification uses `hist_cnt_total_static_host`, a per-capita event *count*, as the disaster-exposure proxy. This section tests whether replacing it with a cumulative *severity* measure changes the result.

**Motivation**: A single high-impact disaster may have a stronger effect on managerial location decisions than many low-impact events. The count proxy treats all events equally; the severity proxy weights each event by the number of people affected, capturing this intensity dimension.

**Variable**: `hist_sev_static_host` = ln(1 + ΣTotalAffected₁₉₈₇₋₂₀₀₇ / pop₁₉₈₇) — time-invariant, pre-period, already on a log scale. It is mean-centred before forming the interaction term, consistent with the main specification.

**Scope**: ENT only; the same three nested models (M1/M2/M3) are estimated for direct comparability with sections 4–5.

In [13]:
# ── Severity-based exposure: already log-scaled in 08_create_sample.py ────────
# hist_sev_static_host = ln(1 + sum(TotalAffected_1987-2007) / pop_1987)
# Mean-centre consistent with main spec to eliminate interaction VIF inflation
df["hist_sev_total_centered"] = (
    df["hist_sev_static_host"] - df["hist_sev_static_host"].mean()
)
df["interaction_ge_hist_sev"] = df["ge_host_centered"] * df["hist_sev_total_centered"]

print("Severity-based exposure — descriptive stats:")
sev_check = ["hist_sev_static_host", "hist_sev_total_centered", "interaction_ge_hist_sev"]
print(df[sev_check].describe().T[["count", "mean", "std", "min", "max"]].round(4))

Severity-based exposure — descriptive stats:
                            count    mean    std     min    max
hist_sev_static_host    2472.0000  0.1976 0.2980  0.0000 1.3612
hist_sev_total_centered 2472.0000  0.0000 0.2980 -0.1976 1.1636
interaction_ge_hist_sev 2472.0000 -0.1035 0.2368 -1.7266 0.6695


In [14]:
M3_sev_regressors_vif = M1_regressors + ["hist_sev_total_centered", "interaction_ge_hist_sev"]

vif_sev_cols = ["ln_avg_ent"] + M3_sev_regressors_vif
vif_sev_df   = df[vif_sev_cols].dropna()
print(f"VIF (severity spec) computed on {len(vif_sev_df):,} complete observations")

X_vif_sev = sm.add_constant(vif_sev_df[M3_sev_regressors_vif].values, prepend=True)
vif_sev_results = pd.DataFrame({
    "Variable": M3_sev_regressors_vif,
    "VIF": [variance_inflation_factor(X_vif_sev, i + 1)
            for i in range(len(M3_sev_regressors_vif))],
}).set_index("Variable")

vif_sev_results["VIF"]  = vif_sev_results["VIF"].round(4)
vif_sev_results["Flag"] = np.where(vif_sev_results["VIF"] > 5, ">> HIGH VIF", "")

print("\nVariance Inflation Factors — Model 3 (severity spec)")
print("=" * 55)
print(vif_sev_results.to_string())
print("=" * 55)

n_high_sev = (vif_sev_results["VIF"] > 5).sum()
if n_high_sev == 0:
    print("\n✓ No variables flagged (all VIF ≤ 5).")
else:
    flagged_sev = vif_sev_results[vif_sev_results["VIF"] > 5].index.tolist()
    print(f"\n⚠ {n_high_sev} variable(s) flagged (VIF > 5): {flagged_sev}")

export_table(apply_labels(vif_sev_results), "vif_severity", "VIF — Model 3 regressors (severity spec)")

VIF (severity spec) computed on 2,472 complete observations

Variance Inflation Factors — Model 3 (severity spec)
                           VIF         Flag
Variable                                   
ln_gdp_pc_host          5.0677  >> HIGH VIF
ln_pop_host             3.1297             
ln_gdp_pc_home          1.0899             
ln_pop_home             1.0839             
ln_distw                1.6170             
contig                  1.2610             
colony                  1.6004             
comlang_off             1.6024             
landlocked_host         1.1444             
ln_surface_area_host    3.2067             
ge_host_centered        4.8901             
hist_sev_total_centered 2.2393             
interaction_ge_hist_sev 1.8421             

⚠ 1 variable(s) flagged (VIF > 5): ['ln_gdp_pc_host']
Exported 'vif_severity': vif_severity.csv, vif_severity.html


In [15]:
M2_sev_regressors = M1_regressors + ["hist_sev_total_centered"]
M3_sev_regressors = M2_sev_regressors + ["interaction_ge_hist_sev"]

res_sev_m1 = run_ols_hc3(df, "ln_avg_ent", M1_regressors)   # same M1 as main spec
res_sev_m2 = run_ols_hc3(df, "ln_avg_ent", M2_sev_regressors)
res_sev_m3 = run_ols_hc3(df, "ln_avg_ent", M3_sev_regressors)

print("Severity models — dep. var: ln_avg_ent (2008–2019 average):")
print(f"  M1:     N={int(res_sev_m1.nobs):,}  Adj.R²={res_sev_m1.rsquared_adj:.4f}")
print(f"  M2-sev: N={int(res_sev_m2.nobs):,}  Adj.R²={res_sev_m2.rsquared_adj:.4f}")
print(f"  M3-sev: N={int(res_sev_m3.nobs):,}  Adj.R²={res_sev_m3.rsquared_adj:.4f}")

Severity models — dep. var: ln_avg_ent (2008–2019 average):
  M1:     N=2,472  Adj.R²=0.6098
  M2-sev: N=2,472  Adj.R²=0.6096
  M3-sev: N=2,472  Adj.R²=0.6129


In [17]:
DISPLAY_ORDER_SEV = [
    "const",
    "ln_gdp_pc_host",        "ln_pop_host",
    "ln_gdp_pc_home",        "ln_pop_home",
    "ln_distw",              "contig",         "colony",        "comlang_off",
    "landlocked_host",       "ln_surface_area_host",
    "ge_host_centered",
    "hist_sev_total_centered",
    "interaction_ge_hist_sev",
]

print(make_stargazer(
    [res_sev_m1, res_sev_m2, res_sev_m3],
    ["Model 1", "Model 2 (sev)", "Model 3 (sev)"],
    var_order=DISPLAY_ORDER_SEV,
    dep_var_label="ln_avg_ent  (2008–19 average) — severity proxy",
))

df_stargazer_sev = stargazer_df(
    [res_sev_m1, res_sev_m2, res_sev_m3],
    ["Model 1", "Model 2 (sev)", "Model 3 (sev)"],
    var_order=DISPLAY_ORDER_SEV,
    dep_var_label="ln_avg_ent (2008–19 average) — severity proxy",
)
export_table(df_stargazer_sev, "regression_severity_ln_avg_ent",
             "Alternative Exposure Proxy (Severity) — ln_avg_ent")

# ── Build the combined, browser-ready file with every table above ────────────
master_path = build_master_html()
webbrowser.open(master_path.resolve().as_uri())

Dep. var: ln_avg_ent  (2008–19 average) — severity proxy
──────────────────────────────────────────────────────────────────────────────────────────────────
                                                   Model 1       Model 2 (sev)       Model 3 (sev)
──────────────────────────────────────────────────────────────────────────────────────────────────
const                                          -22.7654***         -22.7573***         -22.3836***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_host                                   0.3051***           0.3029***           0.2805***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_pop_host                                      0.3617***           0.3618***           0.3664***
                                                  (0.0000)            (0.0000)            (0.0000)
ln_gdp_pc_home                                   1.0

True